In [4]:
import torchvision.models as visionmodels
import torch.nn as nn
import torch.optim as optim
import torch

class Emolens(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = visionmodels.efficientnet_b0(weights=visionmodels.EfficientNet_B0_Weights.DEFAULT)

        for parameter in self.model.parameters():
            parameter.requires_grad = False

        for parameter in self.model.features[-3].parameters():
            parameter.requires_grad = True

        self.model.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(1280, 7))

    def forward(self, x):
        x = self.model(x)

        return x

In [5]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

In [6]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('mv kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')
os.system('kaggle datasets download -d msambare/fer2013 -p /content/ --unzip')

Mounted at /content/drive


0

In [7]:
train_Data = '/content/train'
Test_Data = '/content/test'
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_dataset = datasets.ImageFolder(root=train_Data, transform=train_transform)
test_dataset = datasets.ImageFolder(root=Test_Data, transform=test_transform)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [8]:
model = Emolens()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss()
model_parameters = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.Adam(model_parameters, lr=0.0001)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 132MB/s]


In [9]:
num_of_epochs=10

for epoch in range(num_of_epochs):
    sum_of_train_losses = 0
    sum_of_test_losses = 0
    total_correct = 0
    total_images = 0

    model.train()
    for images, labels in train_dataloader:
        images = images.to(device)
        labels = labels.to(device)
        
        y_pred = model(images)
        loss = criterion(y_pred, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_of_train_losses += loss.item()

    model.eval()
    with torch.no_grad():
        for images, labels in test_dataloader:
            images = images.to(device)
            labels = labels.to(device)
            
            y_pred = model(images)
            loss = criterion(y_pred, labels)

            y_pred = torch.argmax(y_pred, dim=1)
            total_correct += (y_pred == labels).sum().item()
            total_images += labels.size(0)

            sum_of_test_losses += loss.item()

    accuracy = (total_correct / total_images) * 100
    print(f'Epoch {epoch+1}: Train Loss: {sum_of_train_losses / len(train_dataloader): .3f} | Test Loss: {sum_of_test_losses / len(test_dataloader): .3f} | Accuracy: {accuracy: .2f}%')

Epoch 1: Train Loss:  1.424 | Test Loss:  1.156 | Accuracy:  56.09%
Epoch 2: Train Loss:  1.122 | Test Loss:  1.039 | Accuracy:  60.37%
Epoch 3: Train Loss:  1.001 | Test Loss:  0.996 | Accuracy:  62.87%
Epoch 4: Train Loss:  0.912 | Test Loss:  0.970 | Accuracy:  63.42%
Epoch 5: Train Loss:  0.841 | Test Loss:  0.966 | Accuracy:  64.67%
Epoch 6: Train Loss:  0.769 | Test Loss:  0.976 | Accuracy:  64.49%
Epoch 7: Train Loss:  0.704 | Test Loss:  0.978 | Accuracy:  65.32%
Epoch 8: Train Loss:  0.644 | Test Loss:  0.985 | Accuracy:  66.10%
Epoch 9: Train Loss:  0.596 | Test Loss:  1.025 | Accuracy:  65.35%
Epoch 10: Train Loss:  0.542 | Test Loss:  1.045 | Accuracy:  65.80%
